In [1]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.prebuilt import create_react_agent
from qdrant_client import QdrantClient

load_dotenv()

import pdfmux
from langchain_pdfmux import PDFMuxLoader


c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


### Without RAG

In [2]:
MODEL = "qwen3.8-flash"  # cheap + good for learning
llm = ChatOpenAI(model=MODEL, 
                temperature=0,
                base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1",
                extra_body={
                    "enable_thinking": False,
                })

SYSTEM_PROMPT = """You are Tilik, a helpful statistics assistant for Indonesian BPS data.
Rules:
- If you need a number from our sample dataset, call the tool; do not invent values.
- Answer in clear English unless the user writes in Indonesian.
- Keep answers short (2-4 sentences) unless asked for detail.
"""

USER_QUESTION = "Provinsi di indonesia dengan tingkat pengangguran terbuka tertinggi?"

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=USER_QUESTION),
]

llm.invoke(messages)

AIMessage(content='I do not have access to a live database or specific sample dataset in this context to retrieve the current real-time statistics. Therefore, I cannot provide the exact province with the highest open unemployment rate without calling a specific tool that is not available here.\n\nPlease consult the latest official data from BPS (Badan Pusat Statistik) for the most accurate and up-to-date figures.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 97, 'total_tokens': 172, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 97}}, 'model_name': 'qwen3.8-flash', 'system_fingerprint': None, 'id': 'chatcmpl-529b820a-10d7-4806-843d-2c21867bf7b0', 'service_tier': None, 'finish_reason': 'stop', 'logprobs': None}, id='run--01a0bf94-b60d-7610-ac13-f32d7ba8e1b2-0', usage_metadata={'input_tokens': 97, 'output_tokens': 75, 'total_tokens': 172, 'input_tok

### With RAG

In [3]:
loader = PDFMuxLoader("..\data\docs\statistik-demografi-indonesia--hasil-sensus-penduduk-2020.pdf", quality="standard")
docs = loader.load()

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\USER\AppData\Local\Temp\ipykernel_4672\2553987970.py:1: SyntaxWarning: invalid escape sequence '\d'
  loader = PDFMuxLoader("..\data\docs\statistik-demografi-indonesia--hasil-sensus-penduduk-2020.pdf", quality="standard")
[INFO] 2026-09-20 23:09:55,811 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-20 23:09:55,883 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-09-20 23:09:55,884 [RapidOCR] main.py:63: Using C:\Users\USER\AppData\Local\pypoetry\Cache\virtualenvs\tilik-agent-CdBdN4pA-py3.13\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-09-20 23:09:55,965 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-20 23:09:55,967 [RapidOCR] download_file.py:60: File 

In [7]:
docs

[Document(metadata={'source': '..\\data\\docs\\statistik-demografi-indonesia--hasil-sensus-penduduk-2020.pdf', 'title': 'Page 1', 'page_start': 1, 'page_end': 1, 'tokens': 80200, 'confidence': 0.989}, page_content='Katalog: 2201007\nStatistik\nDemografi\nIndonesia\n(Hasil Sensus Penduduk 2020)\nBADAN PUSAT STATISTIK\n\nKatalog: 2201007\nStatistik\nDeoah\nIndonesia\n(Hasil Sensus Penduduk 2020)\nhttps://www\nBADAN PUSAT STATISTIK\n\nStatistik Demografi Indonesia\n(Hasil Sensus Penduduk 2020)\nKatalog: 2201007\nNomor Publikasi: 04100.25001\nUkuran Buku: 17,6 cm X 25 cm\nJumlah Halaman: xviii+168 halaman\nPenyusun Naskah:\nDirektorat Statistik Kependudukan dan Ketenagakerjaan\nPenyunting:\nDirektorat Statistik Kependudukan dan Ketenagakerjaan\nPembuat Kover:\nDirektorat Diseminasi Statistik\nPenerbit:\n©Badan Pusat Statistik\nDicetak oleh:\nBadan Pusat Statistik\nSumber Ilustrasi:\nfreepik.com, canva.com\nDilarang mereproduksi dan/atau menggandakan sebagian atau\nseluruh isi buku ini untu

In [9]:
for i, doc in enumerate(docs):
    print(f"\n=== DOCUMENT {i} ===")
    print(doc.metadata)


=== DOCUMENT 0 ===
{'source': '..\\data\\docs\\statistik-demografi-indonesia--hasil-sensus-penduduk-2020.pdf', 'title': 'Page 1', 'page_start': 1, 'page_end': 1, 'tokens': 80200, 'confidence': 0.989}


In [4]:
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
# Same object for QdrantVectorStore — no wrapper
dim = len(embeddings.embed_query("Apa itu TPK?"))
print(EMBEDDING_MODEL, "dim =", dim)  # 384

sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 dim = 384


In [5]:
client = QdrantClient(url="http://localhost:6333")

vector_store = QdrantVectorStore.

SyntaxError: invalid syntax (3367717077.py, line 3)